In [5]:

import random, pulp as pl

random.seed(42)


drivers = [f"D{i}" for i in range(1,16)]                     
trips   = [f"T{j}" for j in range(1,13)]                     

duration = {t: random.randint(25,70) for t in trips}

base_cost = {d: random.randint(8,15) for d in drivers}
alpha     = {d: random.uniform(0.4,0.6) for d in drivers}

cost = {}
for d in drivers:
    for t in trips:
        if random.random() < 0.25:
            cost[(d,t)] = 1_000_000
        else:
            c = base_cost[d] + alpha[d]*duration[t] + random.uniform(-2,2)
            cost[(d,t)] = round(c,1)

max_trips = {
    "D1": 2, "D2": 2, "D3": 1, "D4": 2, "D5": 3,
    "D6": 2, "D7": 2, "D8": 1, "D9": 2, "D10": 2,
    "D11": 1, "D12": 1, "D13": 1, "D14": 2, "D15": 1,
}

max_time = {d: 8*60 for d in drivers}


mdl = pl.LpProblem("Trip_Assignment", pl.LpMinimize)

# x[d,t] = 1  if driver d performs trip t
x = pl.LpVariable.dicts("x", (drivers, trips), cat="Binary")

# objective
mdl += pl.lpSum(cost[d,t] * x[d][t] for d in drivers for t in trips)

# each trip assigned once
for t in trips:
    mdl += pl.lpSum(x[d][t] for d in drivers) == 1

# Max-Trip per driver
for d in drivers:
    mdl += pl.lpSum(x[d][t] for t in trips) <= max_trips[d]

# Max-Time per driver
for d in drivers:
    mdl += pl.lpSum(duration[t]*x[d][t] for t in trips) <= max_time[d]

mdl.solve(pl.PULP_CBC_CMD(msg=False))

print("Status :", pl.LpStatus[mdl.status])
print("Min-Cost:", pl.value(mdl.objective))
print("-"*60)
for d in drivers:
    assigned = [t for t in trips if x[d][t].value() > 0.5]
    if assigned:
        tot_time = sum(duration[t] for t in assigned)
        tot_cost = sum(cost[d,t] for t in assigned)
        print(f"{d:3} | trips={assigned} | time={tot_time:3} min | cost={tot_cost:.1f}")


Status : Optimal
Min-Cost: 340.5
------------------------------------------------------------
D2  | trips=['T2', 'T6'] | time= 71 min | cost=44.1
D4  | trips=['T5', 'T12'] | time=102 min | cost=62.7
D5  | trips=['T1', 'T4', 'T10'] | time=166 min | cost=99.1
D6  | trips=['T7', 'T8'] | time= 64 min | cost=47.1
D13 | trips=['T9'] | time= 68 min | cost=42.0
D14 | trips=['T3', 'T11'] | time= 56 min | cost=45.5
